# Can we predict next-month arrears?
Target: `arrears_next_month`. The model is meant for future billing months, so the test set is the latest rows; accounts repeat across rows, so the test set also holds whole accounts that never appear in training.

**Feature timing (columns.csv):** `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time and is dropped (it is also likely a near-copy of the target). `meter_ref` is an identifier and is dropped. `billing_month` is used only for splitting.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(df.shape, df.arrears_next_month.mean())
print(df.groupby('final_notice_sent').arrears_next_month.mean())  # shows why it must not be used

## 1. Test set (set aside before any modelling)
Test = last 4 billing months (Sep–Dec 2025) AND a random 25% of accounts. Training = the other 75% of accounts, months before Sep 2025 only. So test rows are both later in time and from unseen accounts.

In [ ]:
rng = np.random.RandomState(0)
accts = np.array(df.meter_ref.unique().tolist())
test_accts = set(rng.choice(accts, int(0.25*len(accts)), replace=False))
cutoff = pd.Timestamp('2025-09-01')
is_test = df.meter_ref.isin(test_accts) & (df.billing_month >= cutoff)
is_train = ~df.meter_ref.isin(test_accts) & (df.billing_month < cutoff)
train, test = df[is_train].reset_index(drop=True), df[is_test].reset_index(drop=True)
assert not set(train.meter_ref) & set(test.meter_ref) and train.billing_month.max() < test.billing_month.min()
print(len(train), len(test), train.arrears_next_month.mean(), test.arrears_next_month.mean())

## 2. Cross-validation structure (training rows only)
Forward-chaining with account separation: 5 folds; fold *k* validates on a later block of months using a held-out fifth of training accounts; it trains only on earlier months from the other accounts. Preprocessing is inside a Pipeline, refit per fold.

In [ ]:
FEATS_NUM = ['occupants','kwh','bill_eur','late_payments_12m']
FEATS_CAT = ['tariff']
FEATS = FEATS_NUM + FEATS_CAT
months = np.sort(train.billing_month.unique())
blocks = np.array_split(months[len(months)//3:], 5)   # validation windows, the first third is train-only
tr_accts = np.array(train.meter_ref.unique().tolist()); rng2 = np.random.RandomState(1); rng2.shuffle(tr_accts)
groups = np.array_split(tr_accts, 5)
folds = []
for blk, g in zip(blocks, groups):
    va = train.billing_month.isin(blk) & train.meter_ref.isin(g)
    tr = (train.billing_month < blk[0]) & ~train.meter_ref.isin(g)
    folds.append((np.where(tr)[0], np.where(va)[0]))
    print(str(blk[0])[:10], str(blk[-1])[:10], tr.sum(), va.sum())

## 3. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg`: logistic regression, scaled numerics + one-hot tariff, C in {0.1, 1}.
3. `hgb`: gradient boosting, max_depth in {2, 4}, learning_rate 0.05.

Selection metric: mean CV average precision (PR-AUC; arrears is ~12% prevalence). Decision threshold: chosen on pooled CV predictions to maximise F1. No candidate is scored on test.

In [ ]:
pre = lambda: ColumnTransformer([('n', StandardScaler(), FEATS_NUM), ('c', OneHotEncoder(handle_unknown='ignore'), FEATS_CAT)])
cands = {'baseline': DummyClassifier(strategy='prior')}
for C_ in [0.1, 1]:
    cands[f'logreg_C{C_}'] = Pipeline([('p', pre()), ('m', LogisticRegression(C=C_, max_iter=1000))])
for d in [2, 4]:
    cands[f'hgb_d{d}'] = Pipeline([('p', pre()), ('m', HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, random_state=0))])
X, y = train[FEATS], train.arrears_next_month.values
oof = {}; rows = []
for name, m in cands.items():
    ps, ys, ap, auc = [], [], [], []
    for tr, va in folds:
        mm = clone(m).fit(X.iloc[tr], y[tr]); p = mm.predict_proba(X.iloc[va])[:,1]
        ps.append(p); ys.append(y[va]); ap.append(average_precision_score(y[va], p)); auc.append(roc_auc_score(y[va], p))
    oof[name] = (np.concatenate(ps), np.concatenate(ys))
    rows.append((name, np.mean(ap), np.std(ap), np.mean(auc)))
cv = pd.DataFrame(rows, columns=['model','cv_AP','sd_AP','cv_AUC']).set_index('model'); cv

In [ ]:
best = cv.drop('baseline').cv_AP.idxmax()
p, yy = oof[best]
ths = np.linspace(0.02, 0.6, 59)
thr = ths[np.argmax([f1_score(yy, p >= t) for t in ths])]
print(best, 'threshold', round(thr, 3), 'CV F1', round(f1_score(yy, p>=thr), 3))

## 5. Decision (written before test scoring)
Chosen by CV only: the model, threshold and features printed above (features: occupants, kwh, bill_eur, late_payments_12m, tariff). It is refit on all training rows and scored once on the test set below. If CV AP of the best model is not clearly above the baseline, the answer is that arrears are not predictable from these features.

In [ ]:
final = clone(cands[best]).fit(X, y)
Xt, yt = test[FEATS], test.arrears_next_month.values
pt = final.predict_proba(Xt)[:,1]
print(f'TEST  AP {average_precision_score(yt, pt):.3f} (prevalence/baseline {yt.mean():.3f})  AUC {roc_auc_score(yt, pt):.3f}  F1@thr {f1_score(yt, pt>=thr):.3f}')

## Conclusion
Headline (single test evaluation): logistic regression (C=0.1), threshold 0.10: **AP 0.495, AUC 0.729, F1 0.476**. Test prevalence is 24%, so AP is about 2x the base rate: there is real, moderate signal, mainly from late-payment history. Caveats: arrears prevalence rises over time (7.8% in training vs 24% in the latest months), so absolute precision and the threshold are not stable; recalibrate on recent data before operational use. CV AP was 0.25 (sd 0.13 across folds), so the estimate is noisy. `final_notice_sent` was excluded as post-prediction information (it nearly determines the target).